### Step 0: Download required Tahoe-100M metadata

In [ ]:
# ! pip install unimol_tools

In [ ]:
from wppkg.dl import hf_download

hf_download(
    repo_id="tahoebio/Tahoe-100M",
    repo_type="dataset",
    allow_patterns=[
        "metadata/obs_metadata.parquet",  # drug dose
        "metadata/drug_metadata.parquet",  # drug SMILES and target
        "metadata/cell_line_metadata.parquet",  # cell line CVCL ID and common name
        "metadata/gene_metadata.parquet",  # gene symbol and Ensembl ID
    ],
    local_dir="./",
)

In [1]:
import pandas as pd

# `cell_line` is the standard ID (CVCL), `cell_name` is the common name
# Pretraining uses `cell_line` as unique identifier to avoid alias ambiguity
obs_metadata = pd.read_parquet("./metadata/obs_metadata.parquet")
obs_metadata["drug"] = obs_metadata["drug"].str.strip()
obs_metadata.head()

,plate,BARCODE_SUB_LIB_ID,sample,gene_count,tscp_count,mread_count,drugname_drugconc,drug,cell_line,sublibrary,BARCODE,pcnt_mito,S_score,G2M_score,phase,pass_filter,cell_name
0,plate10,01_001_001-lib_1681,smp_2359,1379,2172,2559,"[('Bestatin (hydrochloride)', 0.05, 'uM')]",Bestatin (hydrochloride),CVCL_1478,lib_1681,01_001_001,0.029926,-0.229665,-0.190110,G1,full,NCI-H1573
1,plate10,01_002_149-lib_1681,smp_2359,975,1256,1470,"[('Bestatin (hydrochloride)', 0.05, 'uM')]",Bestatin (hydrochloride),CVCL_0459,lib_1681,01_002_149,0.026274,-0.167578,-0.132784,G1,full,NCI-H460
2,plate10,01_003_052-lib_1681,smp_2359,865,1239,1446,"[('Bestatin (hydrochloride)', 0.05, 'uM')]",Bestatin (hydrochloride),CVCL_C466,lib_1681,01_003_052,0.033898,-0.200957,-0.161538,G1,full,hTERT-HPNE
3,plate10,01_003_090-lib_1681,smp_2359,393,484,559,"[('Bestatin (hydrochloride)', 0.05, 'uM')]",Bestatin (hydrochloride),CVCL_1724,lib_1681,01_003_090,0.037190,-0.052746,-0.076190,G1,minimal,SW48
4,plate10,01_003_093-lib_1681,smp_2359,2657,5325,6269,"[('Bestatin (hydrochloride)', 0.05, 'uM')]",Bestatin (hydrochloride),CVCL_1285,lib_1681,01_003_093,0.017465,-0.636364,-0.614103,G1,full,HOP62


In [2]:
drug_metadata = pd.read_parquet("./metadata/drug_metadata.parquet")
drug_metadata = drug_metadata.dropna(subset=["canonical_smiles"])  # drop drugs with missing SMILES
drug_metadata["drug"] = drug_metadata["drug"].str.strip()
drug_metadata["canonical_smiles"] = drug_metadata["canonical_smiles"].str.strip()
drug_metadata.head()

,drug,targets,moa-broad,moa-fine,human-approved,clinical-trials,gpt-notes-approval,canonical_smiles,pubchem_cid
0,Talc,None,unclear,unclear,yes,yes,Talc used in pharma and cosmetics; safety unde...,[OH-].[OH-].[O-][Si]12O[Si]3(O[Si](O1)(O[Si](O...,165411828.0
1,Bortezomib,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma and mantle cell ...,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,387447.0
2,Ixazomib,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma treatment.,B(C(CC(C)C)NC(=O)CNC(=O)C1=C(C=CC(=C1)Cl)Cl)(O)O,25183872.0
3,Ixazomib citrate,"PSMB1, PSMB2, PSMB5",inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma treatment as par...,B1(OC(=O)C(O1)(CC(=O)O)CC(=O)O)C(CC(C)C)NC(=O)...,56844015.0
4,Lactate (calcium),None,unclear,unclear,yes,yes,"Used in medical settings, but not specifically...",C.CC(C(=O)[O-])O.[Ca+2],168311648.0


In [3]:
cell_line_metadata = pd.read_parquet("./metadata/cell_line_metadata.parquet")
cell_line_metadata.head()

,cell_name,Cell_ID_DepMap,Cell_ID_Cellosaur,Organ,Driver_Gene_Symbol,Driver_VarZyg,Driver_VarType,Driver_ProtEffect_or_CdnaEffect,Driver_Mech_InferDM,Driver_GeneType_DM
0,A549,ACH-000681,CVCL_0023,Lung,CDKN2A,Hom,Deletion,DEL,LoF,Suppressor
1,A549,ACH-000681,CVCL_0023,Lung,CDKN2B,Hom,Deletion,DEL,LoF,Suppressor
2,A549,ACH-000681,CVCL_0023,Lung,KRAS,Hom,Missense,p.G12S,GoF,Oncogene
3,A549,ACH-000681,CVCL_0023,Lung,SMARCA4,Hom,Frameshift,p.Q729fs,LoF,Suppressor
4,A549,ACH-000681,CVCL_0023,Lung,STK11,Hom,Stopgain,p.Q37*,LoF,Suppressor


In [4]:
gene_metadata = pd.read_parquet("./metadata/gene_metadata.parquet")
gene_metadata.head()

,gene_symbol,ensembl_id,token_id
0,TSPAN6,ENSG00000000003,3
1,TNMD,ENSG00000000005,4
2,DPM1,ENSG00000000419,5
3,SCYL3,ENSG00000000457,6
4,C1orf112,ENSG00000000460,7


### Step 1: Extract Uni-Mol2 embeddings from drug SMILES

- Uni-Mol2 is a large model by DP Technology for drug SMILES representation

- With 1.1B parameters, Uni-Mol2 encodes a drug into a 1536-dim vector

- Refer to: https://unimol.readthedocs.io/en/latest/quickstart.html#uni-mol-molecule-and-atoms-level-representation

In [5]:
from wppkg import write_json

# Build drug name to SMILES mapping
drug_smiles_dict = dict(zip(drug_metadata["drug"], drug_metadata["canonical_smiles"]))
# Control cells in obs_metadata are labeled 'DMSO_TF'. 
# As it is missing from drug_metadata, we manually added it.
drug_smiles_dict["DMSO_TF"] = "CSC(C)=O"  # DMSO SMILES
write_json(drug_smiles_dict, "./metadata/drug_smiles_dict.json")  # save

len(drug_smiles_dict)

378

In [6]:
import torch
from unimol_tools import UniMolRepr

clf = UniMolRepr(
    data_type='molecule', 
    remove_hs=False,
    model_name='unimolv2',  # avaliable: unimolv1, unimolv2
    model_size='1.1B',  # work when model_name is unimolv2. avaliable: 84m, 164m, 310m, 570m, 1.1B.
    batch_size=32
)

unimol_repr = clf.get_repr(list(drug_smiles_dict.values()), return_atomic_reprs=True)

# CLS token repr
drug_embed = torch.tensor(unimol_repr['cls_repr'], dtype=torch.float32)
drug_embed.shape

2026-07-20 10:21:12 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /data/home/wupengpeng/miniconda3/envs/descope/lib/python3.10/site-packages/unimol_tools/weights/modelzoo/1.1B/checkpoint.pt
2026-07-20 10:22:04 | unimol_tools/data/conformer.py | 459 | INFO | Uni-Mol Tools | Start generating conformers...
378it [00:18, 20.51it/s]
2026-07-20 10:22:28 | unimol_tools/data/conformer.py | 473 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2026-07-20 10:22:28 | unimol_tools/data/conformer.py | 490 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 99.47% of molecules.
2026-07-20 10:22:28 | unimol_tools/data/conformer.py | 499 | INFO | Uni-Mol Tools | Failed 3d conformers indices: [120, 282]
2026-07-20 10:22:28 | unimol_tools/tasks/trainer.py | 78 | INFO | Uni-Mol Tools | Number of GPUs available: 8
2026-07-20 10:22:28 | unimol_tools/tasks/trainer.py | 98 | INFO | Uni-Mol Tools | Using sing

torch.Size([378, 1536])

### Step 2: Append drug dose to Uni-Mol2 embedding

- Tahoe-100M has 4 dose levels: 0.0μM, 0.05μM, 0.5μM, 5.0μM; control cells (DMSO-TF) are marked as 0.0μM

- For private test data: convert dose to μM, then divide by 5 (max-normalization to match pretraining)

- Private data dose should be ≤5μM; control cells should have dose = 0

In [7]:
import ast

unique_drug_dose = obs_metadata["drugname_drugconc"].unique()
dose_values = [float(ast.literal_eval(x)[0][1]) for x in unique_drug_dose]

# Check available dose levels
unique_dose = list(set(dose_values))
unique_dose

[0.05, 0.0, 5.0, 0.5]

In [8]:
from tqdm.auto import tqdm

max_dose = max(unique_dose)

# Append dose at last dimension, get all drug-dose combinations
results = {}  # {"drugname_dose": drug_embed_with_dose}
for drug, embed in tqdm(zip(drug_smiles_dict.keys(), drug_embed), total=len(drug_smiles_dict)):
    for d in unique_dose:
        if drug != "DMSO_TF" and d == 0:
            continue  # skip zero dose for non-control drugs
        elif drug == "DMSO_TF" and d != 0:
            continue  # skip non-zero dose for control drug
        else:
            norm_dose = float(d / max_dose)
            results[f"{drug}_{norm_dose}"] = torch.cat([embed, torch.tensor([norm_dose], dtype=torch.float32)])

torch.save(results, "./metadata/tahoe100m_drug_dose_embed.pt")

  0%|          | 0/378 [00:00<?, ?it/s]